# Demo: Retrieval y Sistema Agéntico

Este notebook demuestra las diferentes estrategias de retrieval y el sistema agéntico.

In [1]:
import sys
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv("../env/.env")

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.retrieval.vector_retriever import VectorRetriever
from graphrag.retrieval.hybrid_retriever import HybridRetriever
from graphrag.retrieval.fulltext_retriever import FullTextRetriever
from graphrag.retrieval.text2cypher import Text2CypherRetriever
from graphrag.agents import AgenticRAG

/home/Pablo/Universidad/02-segundo-cuatrimestre/IAC/GraphRAG-IAC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Inicializar conexión

In [2]:
neo4j = Neo4jManager()

## 2. Probar Vector Retrieval

In [14]:
vector_retriever = VectorRetriever(neo4j)

results = vector_retriever.retrieve("What is the average lifespan of a lion?")

print("Vector Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text']}")
    print(f"   {result['matched_questions']})")

Vector Search Results:

1. Score: 0.975
   Animal: Lion
Section: Reproduction, Cubs, and Lifespan
Sadly, however, less than half of cubs make it to be a year old and four out of five have died by the time they are two, generally either from animal attacks or starvation. Remarkably though, the female lions in the pride will have their cubs at around the same time and will help to suckle and care for the cubs of other females.  
Lion cubs suckle on milk until they are about six months old and although they won’t begin actively hunting until they are about a year old, lion cubs start to eat meat after 12 weeks or so.  
Like most big cats, lions live about 10 to 15 years. In captivity, lions have lived quite a bit longer than in the wild. In 2016, the Philadelphia Zoo had to euthanize a 25-year-old female lion after it began suffering from limited mobility.
   ['What is the typical lifespan of a lion in the wild?', 'Do lions live longer in captivity than in the wild?', 'What percentage of 

## 3. Probar Full-text Retrieval

In [4]:
fulltext_retriever = FullTextRetriever(neo4j)

results = fulltext_retriever.retrieve("What is the average lifespan of an iguana?")

print("Full text Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text']}")

Full text Search Results:

1. Score: 5.196
   Animal: Iguana
Section: Reproduction, Babies, and Lifespan
These animals can live for 15 to 20 years and even longer if cared for properly. At the same time, the San Diego Zoo indicates that some of them can live as long as 60 years. However, wild iguanas have an average lifespan of eight years.

2. Score: 3.158
   Animal: Woodpecker
Section: Reproduction, Babies, and Lifespan
Once a baby first hatches, it develops quickly and is ready to leave the nest in about 30 days. On average, woodpeckers live between four and 12 years. Some can live up to 30 years if environmental conditions are just right.

3. Score: 3.012
   Animal: Bald Eagle
Section: Reproduction, Young, and Lifespan
It takes an average of 35 days for the chicks to emerge from their eggs sporting a brownish-gray head and tail mottled with white. In the 8 to 14 weeks it takes to gain their full-flight feathers, the juveniles spend a lot of time playing with each other, stretching 

## 4. Probar Hybrid Search Retrieval

In [3]:
hybrid_retriever = HybridRetriever(neo4j)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 12877.22it/s]


In [5]:
results = hybrid_retriever.retrieve("What is the average lifespan of an iguana?")

print("Hybrid Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

Hybrid Search Results:

1. Score: 0.997
   Animal: Iguana
Section: Reproduction, Babies, and Lifespan
These animals can live for 15 to 20 years and even longer if cared for properly. At the same time, the San Diego Zoo indicates that some of t...

2. Score: 0.454
   Animal: Iguana
Section: Reproduction, Babies, and Lifespan
Females can store sperm from previous mates for several years if they cannot find suitable mates when ready to lay eggs.  
Females lay eggs f...

3. Score: 0.023
   Animal: Iguana
Section: Appearance
The different species have various sizes ranging from five to seven feet in length. Their whiplike tails make up about half of their body length. The desert iguana i...

4. Score: 0.016
   Animal: Iguana
Section: Habitat
Green iguanas are arboreal, living at the top ends of forest trees. Juveniles reside lower while mature iguanas live higher up. Living high in the tree canopy allows th...

5. Score: 0.016
   Animal: Iguana
Section: Population
You won’t find an overall 

## 5. Probar Text2Cypher

In [4]:
text2cypher = Text2CypherRetriever(neo4j)

In [5]:
# Añadir ejemplos few-shot
text2cypher.add_few_shot_example(
    "How many Mammal species are there?",
    "MATCH (s:Species)-[:BELONGS_TO_CLASS]->(c:AnimalClass {type: 'Mammal'}) RETURN count(s) AS totalMammals"
)

# Example A: Direct property retrieval (Factual)
text2cypher.add_few_shot_example(
    "What is the top speed and maximum weight of a Lion?",
    "MATCH (s:Species {name: 'Lion'}) RETURN s.top_speed_kmh, s.weight_max_kg"
)

# Example B: One-hop traversal
text2cypher.add_few_shot_example(
    "What type of diet do tigers have and what do they feed on?",
    "MATCH (s:Species {name: 'Tiger'})-[:HAS_DIET_TYPE]->(d:DietType), (s)-[:FEEDS_ON]->(f:FoodSource) RETURN d.type, f.type"
)

# Example C: Numerical property filtering
text2cypher.add_few_shot_example(
    "Which animals have a lifespan greater than 50 years?",
    "MATCH (s:Species) WHERE s.lifespan_years > 50 RETURN s.name, s.lifespan_years"
)

# Example D: Relationships with properties (Migration)
text2cypher.add_few_shot_example(
    "Where do whales migrate to in the winter?",
    "MATCH (s:Species {name: 'Whale'})-[m:MIGRATES_TO {season: 'winter'}]->(l:Location) RETURN l.type"
)

# Example E: Graph RAG Integration (Entities to Documents)
text2cypher.add_few_shot_example(
    "In which documents are carnivorous animals mentioned?",
    "MATCH (d:DietType {type: 'Carnivore'})<-[:HAS_DIET_TYPE]-(s:Species)<-[:HAS_ENTITY]-(c:Chunk)<-[:HAS_CHUNK]-(doc:Document) RETURN DISTINCT doc.title"
)

# Ejemplo para Carnívoros (PREYS_ON)
text2cypher.add_few_shot_example(
    "What animals do wolves hunt?",
    "MATCH (s:Species {name: 'Wolf'})-[:PREYS_ON]->(prey:Species) RETURN prey.name"
)

text2cypher.add_few_shot_example(
    "What do cows eat?",
    "MATCH (s:Species {name: 'Cow'})-[:FEEDS_ON]->(f:FoodSource) RETURN f.type"
)

# Ejemplo para Herbívoros (FEEDS_ON)
text2cypher.add_few_shot_example(
    "What kind of plants do elephants feed on?",
    "MATCH (s:Species {name: 'Elephant'})-[:FEEDS_ON]->(f:FoodSource) RETURN f.type"
)

# Ejemplo para Geografía (FOUND_IN)
text2cypher.add_few_shot_example(
    "In which countries or regions are kangaroos found?",
    "MATCH (s:Species {name: 'Kangaroo'})-[:FOUND_IN]->(l:Location) RETURN l.type"
)

# Ejemplo para Bioma/Hábitat (INHABITS)
text2cypher.add_few_shot_example(
    "What kind of ecosystem or biome does the polar bear inhabit?",
    "MATCH (s:Species {name: 'Polar Bear'})-[:INHABITS]->(h:Habitat) RETURN h.type"
)

In [6]:
# ---------------------------------------------------------
# 1. Terminology Maps (English)
# ---------------------------------------------------------

# General
text2cypher.add_terminology_map(
    "animal, creature, species", 
    "Refers to the node with label (:Species)"
)

# Diet distinctions (Carnivores vs Herbivores)
text2cypher.add_terminology_map(
    "carnivore, predator, hunts, preys on, kills", 
    "Use the relationship [:PREYS_ON]->(:Species) to represent hunting and eating other animals."
)
text2cypher.add_terminology_map(
    "herbivore, grazes, eats plants, vegetarian, feeds on", 
    "Use the relationship [:FEEDS_ON]->(:FoodSource) to represent the consumption of non-animal food sources."
)

# Location vs Habitat distinctions
text2cypher.add_terminology_map(
    "country, continent, geographic region, area, located in", 
    "Use the relationship [:FOUND_IN]->(:Location) to refer to specific geographical and political places."
)
text2cypher.add_terminology_map(
    "biome, ecosystem, type of environment, terrain, inhabits", 
    "Use the relationship [:INHABITS]->(:Habitat) to refer to the natural biome or habitat type."
)

text2cypher.add_terminology_map(
    "mentioned in the text, according to the documents, sources say", 
    "Traverse from (:Species)<-[:HAS_ENTITY]-(:Chunk)<-[:HAS_CHUNK]-(:Document)"
)

In [11]:
cypher, results = text2cypher.retrieve("Where are penguins?")

print("Generated Cypher:")
print(cypher)
print("\nResults:")
for result in results:
    print(result)

Generated Cypher:
MATCH (s:Species {name: 'Penguin'})-[:FOUND_IN]->(l:Location) RETURN l.type

Results:
{'l.type': 'South Africa'}
{'l.type': 'Australia'}
{'l.type': 'Antarctica'}
{'l.type': 'New Zealand'}
{'l.type': 'Southern Hemisphere'}
{'l.type': 'Falkland Islands'}
{'l.type': 'Equator'}
{'l.type': 'Northern Hemisphere'}
{'l.type': 'Angola'}
{'l.type': 'Argentina'}
{'l.type': 'Chile'}
{'l.type': 'Namibia'}


## 5. Probar Sistema Agéntico

In [3]:
agentic_rag = AgenticRAG(neo4j)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 13467.67it/s]


In [6]:
# Pregunta simple
result = agentic_rag.answer("Can you tell me who the first president of the United States was?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nIterations:", result['final_critique'])

Question: Can you tell me who the first president of the United States was?

Answer: Sorry, this question is outside my scope. I am designed to answer questions exclusively about zoology and animal biology. If you have a question about the animal world, I'm ready to help!

Iterations: {'is_complete': True, 'is_faithful': True, 'missing_info': []}


In [5]:
# Pregunta compleja que requiere agregación
result = agentic_rag.answer("List all locations mentioned in the documents")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTool used:", result['iterations'][-1]['retrieval']['tool'])
print("\nRouting decision:", result['iterations'][-1]['retrieval']['routing_decision']['reasoning'])

Decision:  tool='vector_search' reasoning="The user is asking to 'List all locations mentioned in the documents.' This is a retrieval task that requires scanning the provided documents for specific entities (locations). Since the goal is to extract all mentions of a category (locations) across the entire corpus, a general semantic search (vector_search) is the most appropriate tool to ensure comprehensive retrieval of all relevant passages, which can then be parsed for the locations. While a keyword search might catch obvious names, vector search is better for identifying all instances of 'location' concepts regardless of how they are phrased or if they are mentioned in passing." query='List all locations mentioned in the documents'


KeyboardInterrupt: 

In [ ]:
# Conversación multi-turno
agentic_rag.reset_conversation()

result1 = agentic_rag.answer("Where do iguanas live?")
print("Q1:", result1['question'])
print("A1:", result1['answer'])

result2 = agentic_rag.answer("Which ?")
print("\nQ2:", result2['question'])
print("A2:", result2['answer'])

In [ ]:
neo4j.close()